**Imports**

In [50]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score


**Load Data Set**

In [51]:
labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

df = pd.read_csv('Toxic_Comment_Dataset_PLUS_50K_Context.csv',
    engine='python',
    on_bad_lines='skip')
df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,2e446f0b4c18ce03,so anyone without knowledge about the subject ...,0,0,0,0,0,0
1,1c220458cfa9ea25,", 20 July 2010 (UTC)\n\n I shall do my best to...",0,0,0,0,0,0
2,c512c7de9b968e27,The cases section(Which reads like a See also ...,0,0,0,0,0,0
3,87957876fa5144bc,"""\n\nUmmm, ya, you warned me for the Cyde issu...",0,0,0,0,0,0
4,a040c54be0d3e47c,Please see the media as this is a blatantly ho...,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
104698,e789358477524477,Beyoncé perfume packaging at holography awards:,0,0,0,0,0,0
104699,37907c59aaa071af,"Hoboken Terminal \n\nHey, could you double che...",0,0,0,0,0,0
104700,c78bf4894ac99b80,Thank you for the kind message \n\nThank you f...,0,0,0,0,0,0
104701,3cbde142897210ee,The French and Indian War was not fought betwe...,0,0,0,0,0,0


In [52]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(example):
    encoding = tokenizer(
        example["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    encoding["labels"] = [float(example[label]) for label in labels]
    return encoding

**Train Test Split**

In [53]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)



**Preprocess**

In [54]:
train_dataset = train_dataset.map(preprocess)
val_dataset = test_dataset.map(preprocess)
train_dataset = train_dataset.remove_columns(["comment_text"])
val_dataset = val_dataset.remove_columns(["comment_text"])
train_dataset.set_format("torch")
val_dataset.set_format("torch")

Map:   0%|          | 0/83762 [00:00<?, ? examples/s]

Map:   0%|          | 0/20941 [00:00<?, ? examples/s]

**Load the Model**

In [55]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    problem_type="multi_label_classification"
)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Training Args**

In [56]:
training_args = TrainingArguments(
    output_dir="./toxic_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2, #3
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=True
)

**Train**

In [57]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)
trainer.train()
results = trainer.evaluate()
print(results)

Epoch,Training Loss,Validation Loss
1,0.047708,0.039756
2,0.031643,0.040755


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.04075537994503975, 'eval_runtime': 47.4457, 'eval_samples_per_second': 441.368, 'eval_steps_per_second': 55.179, 'epoch': 2.0}


**Prediction**

In [58]:
predictions = trainer.predict(val_dataset)

probs = 1 / (1 + np.exp(-predictions.predictions))  # sigmoid
preds = (probs >= 0.3).astype(int)
true_labels = predictions.label_ids

f1_weighted = f1_score(true_labels, preds, average="weighted")
print(f"Weighted F1: {f1_weighted * 100:.2f}%")
f1 = f1_score(true_labels, preds, average="macro")
print(f"F1 Score: {f1 * 100:.2f}%")

Weighted F1: 79.29%
F1 Score: 68.96%


**Save Model**

In [59]:
from google.colab import drive
# drive.mount('/content/drive')
# save_path = "/content/drive/MyDrive/toxic_bert_model"

trainer.save_model("./toxic_bert_model")
tokenizer.save_pretrained("./toxic_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./toxic_bert_model/tokenizer_config.json',
 './toxic_bert_model/tokenizer.json')

In [60]:
import shutil
shutil.make_archive("toxic_bert_model", "zip", "./toxic_bert_model")

from google.colab import files

files.download("toxic_bert_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**TEST**

In [61]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

MODEL_PATH = "./toxic_bert_model"

tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [62]:
labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

def detect_toxicity(text, threshold=0.5):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)[0]

    results = {
        label: float(prob)
        for label, prob in zip(labels, probs)
    }

    predicted = [
        label
        for label, prob in results.items()
        if prob >= threshold
    ]

    return predicted, results

In [63]:
pred, resu = detect_toxicity("nigger")
print(pred)

['toxic', 'obscene', 'insult', 'identity_hate']
